# Gemma 2B IT LoRA training

Burstchester dataset ID를 받아 Gemma 2B IT LoRA fine-tuning을 실행하고, adapter를 Hugging Face에 업로드한 뒤 Burstchester 모델로 등록한다.

In [ ]:
# 1. Clone the CLI repository.
!git clone https://github.com/tomongoose/burstchester.git /content/burstchester
%cd /content/burstchester

In [ ]:
# 2. Load secrets from Colab secrets first, then fall back to hidden input.
import os
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None

def secret(name, prompt=None, required=True):
    value = os.environ.get(name)
    if not value and userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value and prompt:
        value = getpass(prompt)
    if required and not str(value or '').strip():
        raise ValueError(f'{name} is required.')
    if value:
        os.environ[name] = str(value).strip()
    return os.environ.get(name, '')

secret('BURSTCHESTER_ACCESS_TOKEN', 'Burstchester access token: ')
secret('HF_TOKEN', 'Hugging Face token: ')
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
print('Secrets configured in environment.')

In [ ]:
# 3. Configure Gemma 2B IT LoRA.
import os

# Change this value to train on different Burstchester datasets.
# Use commas, spaces, or new lines to provide multiple dataset IDs.
DATASET_IDS = 'dataset-id-1,dataset-id-2'

if not DATASET_IDS.strip() or DATASET_IDS == 'dataset-id-1,dataset-id-2':
    raise ValueError('Set DATASET_IDS to your Burstchester dataset ID(s) before training.')

os.environ['DATASET_IDS'] = DATASET_IDS
os.environ['TRAIN_COMMAND'] = 'train-gemma-2b-it-lora'
os.environ['BASE_MODEL'] = 'google/gemma-2b-it'
os.environ['WORKSPACE'] = '/content/burstchester-training/gemma-2b-it-lora'
os.environ['OUTPUT_MODEL_REPO'] = 'hf-user/gemma-2b-it-lora'  # Change this.
os.environ['TRAINING_METHOD'] = 'lora'
os.environ['EPOCHS'] = '1'
os.environ['BATCH_SIZE'] = '1'
os.environ['MAX_SEQ_LENGTH'] = '128'
os.environ['LORA_RANK'] = '8'
os.environ['LORA_ALPHA'] = '16'
os.environ['LORA_DROPOUT'] = '0.05'
os.environ['MODEL_POINT_COST'] = '30'

print('Training dataset IDs:', os.environ['DATASET_IDS'])

In [ ]:
# 4. Run training, upload, and registration.
!bash cli/scripts/colab-train-and-register.sh